# 02_silver_to_gold: BikeStores
Builds business-ready Gold tables from Silver: fact_sales, daily and monthly sales,
top products by revenue, customer lifetime value and sales by store.
Logs rows read, rejected and written for each Gold table.

In [0]:
from datetime import datetime
from pyspark.sql import functions as F, Window

c = F.col

STORAGE   = "cdestorageacc"
CONTAINER = "onpremsqlserver"
BASE      = f"abfss://{CONTAINER}@{STORAGE}.dfs.core.windows.net"
SILVER    = f"{BASE}/silver"
GOLD      = f"{BASE}/gold"
LOGS      = f"{BASE}/logs"

def silver(table):
    return spark.read.format("delta").load(f"{SILVER}/{table.replace('.', '/')}")

orders     = silver("sales.orders")
items      = silver("sales.order_items")
customers  = silver("sales.customers")
stores     = silver("sales.stores")
products   = silver("production.products")
brands     = silver("production.brands")
categories = silver("production.categories")

log = []

def write_gold(name, df, rows_read, rows_rejected=0):
    written = df.count()
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{GOLD}/{name}")
    entry = dict(run_ts=datetime.utcnow().isoformat(timespec="seconds"), notebook="02_silver_to_gold",
                 table=name, rows_read=rows_read, rows_rejected=rows_rejected, rows_written=written)
    log.append(entry)
    print(f"{name}: read={rows_read} rejected={rows_rejected} written={written}")

## fact_sales (line level)
Order status 3 (Rejected) is excluded from revenue. Order lines with no matching
order, product, brand, category or store are counted as rejected.

In [0]:
items_read = items.count()

orders_ok = orders.filter(c("order_status") != 3)

fact_sales = (
    items
    .join(orders_ok, "order_id", "inner")
    .join(products.select("product_id", "product_name", "brand_id", "category_id"), "product_id", "inner")
    .join(brands, "brand_id", "inner")
    .join(categories, "category_id", "inner")
    .join(stores.select("store_id", F.col("store_name"), F.col("city").alias("store_city"),
                        F.col("state").alias("store_state")), "store_id", "inner")
    .withColumn("revenue", F.round(c("quantity") * c("list_price") * (1 - c("discount")), 2))
    .select("order_id", "item_id", "order_date", "order_status", "customer_id", "store_id", "store_name",
            "store_city", "store_state", "product_id", "product_name", "brand_name", "category_name",
            "quantity", "list_price", "discount", "revenue")
)

fact_count = fact_sales.count()
write_gold("fact_sales", fact_sales, rows_read=items_read, rows_rejected=items_read - fact_count)

fact_sales: read=4722 rejected=123 written=4599


/home/spark-54177081-d0c7-4266-9577-0d/.ipykernel/63/command-6370259885002398-39894243:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  entry = dict(run_ts=datetime.utcnow().isoformat(timespec="seconds"), notebook="02_silver_to_gold",


## Daily and monthly sales summary

In [0]:
daily_sales = (
    fact_sales.groupBy("order_date")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
         F.countDistinct("order_id").alias("orders"),
         F.sum("quantity").alias("units_sold"))
    .orderBy("order_date")
)
write_gold("daily_sales_summary", daily_sales, rows_read=fact_count)

monthly_sales = (
    fact_sales.withColumn("month_start", F.date_trunc("month", c("order_date")).cast("date"))
    .groupBy("month_start")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
         F.countDistinct("order_id").alias("orders"),
         F.sum("quantity").alias("units_sold"))
    .withColumn("year", F.year("month_start"))
    .withColumn("month", F.month("month_start"))
    .orderBy("month_start")
)
write_gold("monthly_sales_summary", monthly_sales, rows_read=fact_count)

/home/spark-54177081-d0c7-4266-9577-0d/.ipykernel/63/command-6370259885002398-39894243:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  entry = dict(run_ts=datetime.utcnow().isoformat(timespec="seconds"), notebook="02_silver_to_gold",


daily_sales_summary: read=4599 rejected=0 written=710


/home/spark-54177081-d0c7-4266-9577-0d/.ipykernel/63/command-6370259885002398-39894243:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  entry = dict(run_ts=datetime.utcnow().isoformat(timespec="seconds"), notebook="02_silver_to_gold",


monthly_sales_summary: read=4599 rejected=0 written=28


## Top products by revenue

In [0]:
top_products = (
    fact_sales.groupBy("product_id", "product_name", "brand_name", "category_name")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
         F.sum("quantity").alias("units_sold"))
    .withColumn("revenue_rank", F.dense_rank().over(Window.orderBy(c("total_revenue").desc())))
    .orderBy("revenue_rank")
)
write_gold("top_products", top_products, rows_read=fact_count)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


top_products: read=4599 rejected=0 written=305


/home/spark-54177081-d0c7-4266-9577-0d/.ipykernel/63/command-6370259885002398-39894243:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  entry = dict(run_ts=datetime.utcnow().isoformat(timespec="seconds"), notebook="02_silver_to_gold",


## Customer lifetime value

In [0]:
customer_ltv = (
    fact_sales.groupBy("customer_id")
    .agg(F.round(F.sum("revenue"), 2).alias("lifetime_value"),
         F.countDistinct("order_id").alias("total_orders"),
         F.min("order_date").alias("first_order_date"),
         F.max("order_date").alias("last_order_date"))
    .withColumn("avg_order_value", F.round(c("lifetime_value") / c("total_orders"), 2))
    .join(customers.select("customer_id", "first_name", "last_name", "city", "state"), "customer_id", "left")
    .orderBy(c("lifetime_value").desc())
)
write_gold("customer_lifetime_value", customer_ltv, rows_read=fact_count)

customer_lifetime_value: read=4599 rejected=0 written=1445


/home/spark-54177081-d0c7-4266-9577-0d/.ipykernel/63/command-6370259885002398-39894243:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  entry = dict(run_ts=datetime.utcnow().isoformat(timespec="seconds"), notebook="02_silver_to_gold",


## Sales by store

In [0]:
sales_by_store = (
    fact_sales.groupBy("store_id", "store_name", "store_city", "store_state")
    .agg(F.round(F.sum("revenue"), 2).alias("total_revenue"),
         F.countDistinct("order_id").alias("orders"),
         F.sum("quantity").alias("units_sold"))
    .orderBy(c("total_revenue").desc())
)
write_gold("sales_by_store", sales_by_store, rows_read=fact_count)

sales_by_store: read=4599 rejected=0 written=3


/home/spark-54177081-d0c7-4266-9577-0d/.ipykernel/63/command-6370259885002398-39894243:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  entry = dict(run_ts=datetime.utcnow().isoformat(timespec="seconds"), notebook="02_silver_to_gold",


## Persist the run log

In [0]:
log_df = spark.createDataFrame(log)
log_df.write.format("delta").mode("append").save(f"{LOGS}/gold_run_log")
display(log_df)

notebook,rows_read,rows_rejected,rows_written,run_ts,table
02_silver_to_gold,4722,123,4599,2026-11-12T08:38:53,fact_sales
02_silver_to_gold,4599,0,710,2026-11-12T08:38:57,daily_sales_summary
02_silver_to_gold,4599,0,28,2026-11-12T08:38:59,monthly_sales_summary
02_silver_to_gold,4599,0,305,2026-11-12T08:39:03,top_products
02_silver_to_gold,4599,0,1445,2026-11-12T08:39:06,customer_lifetime_value
02_silver_to_gold,4599,0,3,2026-11-12T08:39:09,sales_by_store
